# BaSiCPy preprocessing

Interactive notebook version of `scripts/run_basicpy.py`.

Run this notebook **before** `main.ipynb` when you have a new raw time-lapse. It creates the BaSiCPy-corrected TIFF that `main.ipynb` later loads as `BASICPY_TIF_PATH`.

High-level flow:
1. Set input/output paths.
2. Run one preprocessing cell to load the raw stack, optionally register frames, fit BaSiCPy, transform the movie, and save outputs.
3. Run the sanity-check plot.


## 0. Setup

Run this notebook in the separate **`basicpy`** Python environment, not the `ultrack` environment used by `main.ipynb`.

Main packages used here:

- `numpy`
- `tifffile`
- `basicpy`
- `scipy`
- `scikit-image`
- `tqdm`
- `matplotlib`

Why separate environment: BaSiCPy pins dependency versions that can conflict with `ultrack`/`numba`, so this notebook writes corrected TIFFs to disk and `main.ipynb` simply loads those files later.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import tifffile
from basicpy import BaSiC
from scipy.ndimage import shift as ndi_shift
from skimage.registration import phase_cross_correlation
from tqdm import tqdm

import matplotlib.pyplot as plt


## 1. Input/output config

Change these paths/options for a different trial/movie. The main output, `OUTPUT_TIMELAPSE_PATH`, is the file that `main.ipynb` should load as `BASICPY_TIF_PATH`.


In [ ]:
# Project/root paths
# Robust to running the notebook from either the repo root or a subfolder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "images").exists() and (PROJECT_ROOT.parent / "images").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Input/output files: change these for a different trial/movie.
INPUT_TIF_PATH = PROJECT_ROOT / "images" / "stacked.tif"
OUTPUT_DIR = PROJECT_ROOT / "images"
OUTPUT_PREFIX = INPUT_TIF_PATH.stem

OUTPUT_TIMELAPSE_PATH = OUTPUT_DIR / f"{OUTPUT_PREFIX}_basicpy_timelapse.tif"
OUTPUT_FLATFIELD_PATH = OUTPUT_DIR / f"{OUTPUT_PREFIX}_basicpy_flatfield_only.tif"
DIAGNOSTICS_DIR = OUTPUT_DIR / "basicpy_diagnostics"

# Processing options
ALIGN_FRAMES = True          # register frames before BaSiCPy fitting
UPSAMPLE_FACTOR = 10         # sub-pixel registration precision
GET_DARKFIELD = True         # model persistent additive background/pattern
SKIP_AUTOTUNE = False        # True = faster, usually less accurate
SAVE_ALIGNED_STACK = True    # useful diagnostic, can be large

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_TIF_PATH:", INPUT_TIF_PATH)
print("OUTPUT_TIMELAPSE_PATH:", OUTPUT_TIMELAPSE_PATH)
print("OUTPUT_FLATFIELD_PATH:", OUTPUT_FLATFIELD_PATH)
print("DIAGNOSTICS_DIR:", DIAGNOSTICS_DIR)


## 2. Run BaSiCPy preprocessing

This single cell runs the full BaSiCPy preprocessing pipeline:

1. Load `INPUT_TIF_PATH`.
2. Optionally register frames by translation before BaSiCPy fitting (`ALIGN_FRAMES`).
3. Fit BaSiCPy with flatfield/darkfield correction.
4. Save the timelapse-corrected output, flatfield-only comparison output, and diagnostics.

You usually only need to edit the config above, then run this cell.


In [ ]:
# Helper functions
def percentile_normalize(frame: np.ndarray, p_low: float = 1.0, p_high: float = 99.8) -> np.ndarray:
    """Normalize one frame to [0, 1] using robust percentiles (registration only)."""
    lo, hi = np.percentile(frame, (p_low, p_high))
    if hi <= lo:
        return np.zeros_like(frame, dtype=np.float32)
    out = (frame.astype(np.float32) - lo) / (hi - lo)
    return np.clip(out, 0, 1).astype(np.float32)


def register_translation_stack(
    stack: np.ndarray,
    upsample_factor: int = 10,
) -> tuple[np.ndarray, np.ndarray]:
    """Sub-pixel translational registration using phase cross-correlation."""
    reference = np.median(stack, axis=0)
    reference_norm = percentile_normalize(reference)

    aligned = np.empty(stack.shape, dtype=np.float32)
    shifts = np.zeros((stack.shape[0], 2), dtype=np.float32)

    for t, frame in enumerate(tqdm(stack, desc="register")):
        moving_norm = percentile_normalize(frame)
        est_shift, _error, _phase = phase_cross_correlation(
            reference_norm,
            moving_norm,
            upsample_factor=upsample_factor,
            normalization="phase",
        )
        shifts[t] = est_shift[:2]
        aligned[t] = ndi_shift(
            frame.astype(np.float32),
            shift=est_shift[:2],
            order=1,
            mode="nearest",
            prefilter=False,
        )

    return aligned, shifts


def save_image(path: Path, array: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tifffile.imwrite(path, np.asarray(array, dtype=np.float32), photometric="minisblack")


def save_baseline_plot(path: Path, baseline: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(np.asarray(baseline).reshape(-1))
    ax.set_xlabel("Frame")
    ax.set_ylabel("Baseline")
    ax.set_title("BaSiC per-frame baseline (contrast drift)")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.show()


# Load raw stack
stack = tifffile.imread(INPUT_TIF_PATH)
if stack.ndim != 3:
    raise ValueError(f"Expected a 3D (T, Y, X) stack, got shape {stack.shape}")

print(f"Loaded {INPUT_TIF_PATH}: shape={stack.shape}, dtype={stack.dtype}, min={stack.min()}, max={stack.max()}")


# Optional registration before BaSiCPy
shifts: np.ndarray | None = None

if ALIGN_FRAMES:
    print("Registering frames (phase cross-correlation vs. temporal median)...")
    work_stack, shifts = register_translation_stack(stack, upsample_factor=UPSAMPLE_FACTOR)

    DIAGNOSTICS_DIR.mkdir(parents=True, exist_ok=True)
    if SAVE_ALIGNED_STACK:
        save_image(DIAGNOSTICS_DIR / "aligned_stack.tif", work_stack)

    np.savetxt(
        DIAGNOSTICS_DIR / "alignment_shifts_yx.csv",
        shifts,
        delimiter=",",
        header="dy,dx",
        comments="",
    )
    print(
        f"  max |dy|={np.abs(shifts[:, 0]).max():.2f}px, "
        f"max |dx|={np.abs(shifts[:, 1]).max():.2f}px"
    )
else:
    work_stack = stack.astype(np.float32)
    print("Skipping registration; using raw stack for BaSiCPy fitting.")


# Fit BaSiCPy
basic = BaSiC(get_darkfield=GET_DARKFIELD)

if not SKIP_AUTOTUNE:
    print("Autotuning BaSiC hyperparameters (is_timelapse=True)...")
    basic.autotune(work_stack, is_timelapse=True)
else:
    print("Skipping autotune, using BaSiC defaults.")

print("Fitting BaSiC (flatfield/darkfield)...")
basic.fit(work_stack)
print("BaSiCPy fit complete.")


# Transform and save outputs
print("Applying correction (transform, is_timelapse=True)...")
corrected_tl = basic.transform(work_stack, is_timelapse=True)
baseline = None
if isinstance(corrected_tl, tuple):
    corrected_tl_stack, baseline = corrected_tl
else:
    corrected_tl_stack = corrected_tl

print("Applying correction (transform, is_timelapse=False), for comparison...")
corrected_flat = basic.transform(work_stack, is_timelapse=False)
corrected_flat_stack = corrected_flat[0] if isinstance(corrected_flat, tuple) else corrected_flat


def _finalize(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr, dtype=np.float32)
    arr -= np.nanmin(arr)
    return arr


corrected_tl_stack = _finalize(corrected_tl_stack)
corrected_flat_stack = _finalize(corrected_flat_stack)

save_image(OUTPUT_TIMELAPSE_PATH, corrected_tl_stack)
save_image(OUTPUT_FLATFIELD_PATH, corrected_flat_stack)

DIAGNOSTICS_DIR.mkdir(parents=True, exist_ok=True)
save_image(DIAGNOSTICS_DIR / "flatfield.tif", np.asarray(basic.flatfield, dtype=np.float32))
if getattr(basic, "darkfield", None) is not None:
    save_image(DIAGNOSTICS_DIR / "darkfield.tif", np.asarray(basic.darkfield, dtype=np.float32))

if baseline is not None:
    np.savetxt(DIAGNOSTICS_DIR / "timelapse_baseline.csv", np.asarray(baseline).reshape(-1), delimiter=",")
    save_baseline_plot(DIAGNOSTICS_DIR / "timelapse_baseline.png", baseline)
elif getattr(basic, "baseline", None) is not None:
    base_arr = np.asarray(basic.baseline).reshape(-1)
    np.savetxt(DIAGNOSTICS_DIR / "timelapse_baseline.csv", base_arr, delimiter=",")
    save_baseline_plot(DIAGNOSTICS_DIR / "timelapse_baseline.png", base_arr)

metadata = {
    "input": str(INPUT_TIF_PATH),
    "output_timelapse": str(OUTPUT_TIMELAPSE_PATH),
    "output_flatfield_only": str(OUTPUT_FLATFIELD_PATH),
    "shape": list(stack.shape),
    "align": ALIGN_FRAMES,
    "upsample_factor": UPSAMPLE_FACTOR if ALIGN_FRAMES else None,
    "get_darkfield": GET_DARKFIELD,
    "autotuned": not SKIP_AUTOTUNE,
    "output_timelapse_min": float(np.nanmin(corrected_tl_stack)),
    "output_timelapse_max": float(np.nanmax(corrected_tl_stack)),
    "output_flatfield_only_min": float(np.nanmin(corrected_flat_stack)),
    "output_flatfield_only_max": float(np.nanmax(corrected_flat_stack)),
}
if shifts is not None:
    metadata["max_abs_shift_dy"] = float(np.abs(shifts[:, 0]).max())
    metadata["max_abs_shift_dx"] = float(np.abs(shifts[:, 1]).max())

(DIAGNOSTICS_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))

print(f"Wrote corrected stack (timelapse-corrected):     {OUTPUT_TIMELAPSE_PATH}")
print(f"Wrote corrected stack (flatfield/darkfield only): {OUTPUT_FLATFIELD_PATH}")
print(f"Wrote diagnostics: {DIAGNOSTICS_DIR}")


## 3. Sanity Check

Compare raw vs. BaSiCPy outputs on a few full frames. This is only a first-pass check for large illumination drift/background issues; the later ECC+RPCA correction in `main.ipynb` handles the sharp hydrogel pattern.


In [ ]:
check_frames = [0, stack.shape[0] // 4, stack.shape[0] // 2, stack.shape[0] - 1]

fig, axes = plt.subplots(len(check_frames), 3, figsize=(13, 4.5 * len(check_frames)))
for i, t in enumerate(check_frames):
    panels = [
        ("raw", stack[t]),
        ("BaSiCPy timelapse", corrected_tl_stack[t]),
        ("BaSiCPy flatfield only", corrected_flat_stack[t]),
    ]
    for ax, (title, arr) in zip(axes[i], panels):
        lo, hi = np.percentile(arr, (1, 99.5))
        ax.imshow(arr, cmap="gray", vmin=lo, vmax=hi)
        ax.set_title(f"t={t}: {title}", fontsize=10)
        ax.axis("off")

fig.tight_layout()
plt.show()


## Use output in `main.ipynb`

After this notebook finishes, open `main.ipynb` in the `ultrack` environment and set:

```python
BASICPY_TIF_PATH = Path("images/stacked_basicpy_timelapse.tif")
```

or to whatever path is printed as `OUTPUT_TIMELAPSE_PATH` above.
